# 01 · Feature extraction — FakeAVCeleb (voice + face image + ASR transcript)

**CSET 346 · Multimodal NLP for Deepfake Detection** — runs on **Kaggle** (GPU T4/P100).

What this notebook does, once, so every later experiment trains in minutes on saved embeddings:

| Stream | Preprocessing | Frozen encoder | Saved as |
|---|---|---|---|
| Image | 3 frames per clip, face crop (OpenCV Haar, centre-crop fallback) | CLIP ViT-B/32 | `image` (N, 512) |
| Voice | audio track → 16 kHz mono, first 10 s | Wav2Vec2-base, mean-pooled **per layer** | `audio_layers` (N, 13, 768) |
| Text | Whisper-base transcript of the voice | CLIP text encoder | `text` (N, 512) + transcript in `meta.csv` |

**Before you run**
1. *Add Input* → search `shreyaty08/fakeavceleb` → add it.
2. Settings → Accelerator: **GPU T4 x2** (or P100). Settings → **Internet: On** (needed to download CLIP / Wav2Vec2 / Whisper from Hugging Face; it asks for phone verification once).
3. Run all cells. Output goes to `/kaggle/working/features/` — afterwards use *Save Version → Save & Run All* and then *New Dataset* from the output, so `02_exp1_baselines` can read it.

The run is **resumable**: batches are checkpointed to `features/parts/`, so if the session dies, just re-run.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "opencv-python-headless"], check=False)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'opencv-python-headless'], returncode=0)

In [2]:
import os, re, json, hashlib, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ---------------- CONFIG ----------------
INPUT_ROOT = "/kaggle/input"
OUT_DIR = Path(os.environ.get("FEAT_OUT", "/kaggle/working/features"))
SEED = 42

N_REAL = None              # None = use ALL real clips (REAL is the minority class in FakeAVCeleb)
FAKE_TO_REAL_RATIO = 1.0   # 1.0 -> balanced. Fake clips are spread evenly over RVFA / FVRA / FVFA
N_FRAMES = 3               # frames sampled per clip
MAX_AUDIO_SEC = 10         # audio cap per clip
BATCH = 16                 # clips per checkpointed batch (also the Whisper batch size)

CLIP_ID = "openai/clip-vit-base-patch32"
W2V_ID = "facebook/wav2vec2-base"
WHISPER_ID = "openai/whisper-base"
WHISPER_LANG = "en"        # FakeAVCeleb is English. Set to None for the Hindi-English extension.

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1 · Find the videos, parse label / category / speaker from the path

FakeAVCeleb has four categories: **RVRA** real-video/real-audio (the only REAL class), **RVFA** real video + cloned voice,
**FVRA** face-swapped video + real voice, **FVFA** both fake. Binary label: `1 = FAKE` for anything except RVRA.
`video_fake` / `audio_fake` flags are kept for the later ablation and generalization experiments.

In [3]:
VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv"}
files = [p for p in Path(INPUT_ROOT).rglob("*") if p.suffix.lower() in VIDEO_EXT]
print("video files found:", len(files))
for p in files[:5]:
    print("  ", p)

CAT_PATTERNS = {
    "RVRA": r"real[\s_\-]*video[\s_\-]*real[\s_\-]*audio",
    "RVFA": r"real[\s_\-]*video[\s_\-]*fake[\s_\-]*audio",
    "FVRA": r"fake[\s_\-]*video[\s_\-]*real[\s_\-]*audio",
    "FVFA": r"fake[\s_\-]*video[\s_\-]*fake[\s_\-]*audio",
}

def parse_category(path):
    s = str(path).lower()
    for k, pat in CAT_PATTERNS.items():
        if re.search(pat, s):
            return k
    return None

def parse_speaker(path):
    rel = str(Path(path).relative_to(INPUT_ROOT)).lower()
    m = re.search(r"(?<![a-z0-9])(id\d{3,})", rel)
    return m.group(1) if m else "unknown"

meta = pd.DataFrame({"path": [str(p) for p in files]})
meta["category"] = meta.path.map(parse_category)

if meta.category.isna().all():
    raise RuntimeError(
        "Could not detect FakeAVCeleb categories from the folder names. "
        "Print a few paths above and adjust CAT_PATTERNS to match this mirror's naming.\n"
        + "\n".join(meta.path.head(10))
    )
n_unmatched = int(meta.category.isna().sum())
if n_unmatched:
    print(f"WARNING: {n_unmatched} videos had no recognisable category and are dropped")
meta = meta.dropna(subset=["category"]).reset_index(drop=True)

meta["speaker"] = meta.path.map(parse_speaker)
meta["label"] = (meta.category != "RVRA").astype(int)                 # 1 = FAKE
meta["video_fake"] = meta.category.isin(["FVRA", "FVFA"]).astype(int)
meta["audio_fake"] = meta.category.isin(["RVFA", "FVFA"]).astype(int)

print("\nFull dataset counts per category:")
print(meta.category.value_counts().to_string())
print("distinct speakers:", meta.speaker.nunique(), "| 'unknown' speaker rows:", int((meta.speaker == "unknown").sum()))

video files found: 21560
   /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/Caucasian (European)/women/id00042/00028.mp4
   /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/Caucasian (European)/women/id00068/00004.mp4
   /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/Caucasian (European)/women/id00071/00014.mp4
   /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/Caucasian (European)/women/id00188/00020.mp4
   /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/Caucasian (European)/women/id00232/00025.mp4

Full dataset counts per category:
category
FVFA    10851
FVRA     9709
RVFA      500
RVRA      500
distinct speakers: 500 | 'unknown' speaker rows: 0


## 2 · Balanced subsample + speaker-grouped train/val/test split

Splitting **by speaker** (not by clip) stops the same identity appearing in train and test, which would otherwise let
the model memorise people instead of manipulation artefacts. Document this in the paper's dataset section.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

real = meta[meta.label == 0]
if N_REAL:
    real = real.sample(min(N_REAL, len(real)), random_state=SEED)
n_fake = int(round(FAKE_TO_REAL_RATIO * len(real)))
fake_cats = [c for c in ["RVFA", "FVRA", "FVFA"] if (meta.category == c).any()]
per_cat = n_fake // max(len(fake_cats), 1)
fake = pd.concat([
    meta[meta.category == c].sample(min(per_cat, int((meta.category == c).sum())), random_state=SEED)
    for c in fake_cats
])
sub = pd.concat([real, fake]).sample(frac=1, random_state=SEED).reset_index(drop=True)
sub["clip_id"] = np.arange(len(sub))

def grouped_split(df, seed=SEED, val=0.15, test=0.15):
    idx = np.arange(len(df))
    g = df.speaker.values
    trv, te = next(GroupShuffleSplit(1, test_size=test, random_state=seed).split(idx, groups=g))
    tr_i, va_i = next(GroupShuffleSplit(1, test_size=val / (1 - test), random_state=seed).split(trv, groups=g[trv]))
    return trv[tr_i], trv[va_i], te

tr, va, te = grouped_split(sub)
sub["split"] = "train"
sub.loc[va, "split"] = "val"
sub.loc[te, "split"] = "test"

print("Subset size:", len(sub))
print(pd.crosstab(sub.split, sub.category).to_string())
print("\nREAL / FAKE per split:")
print(pd.crosstab(sub.split, sub.label).rename(columns={0: "REAL", 1: "FAKE"}).to_string())
assert set(sub.speaker[sub.split == "test"]).isdisjoint(set(sub.speaker[sub.split != "test"])) or (sub.speaker == "unknown").any(), \
    "speaker leakage between test and train/val"

# resumability: wipe stale checkpoints if the selection changed
fingerprint = hashlib.md5((",".join(sub.path) + f"|{N_FRAMES}|{MAX_AUDIO_SEC}|{CLIP_ID}|{W2V_ID}|{WHISPER_ID}").encode()).hexdigest()
fp_file = OUT_DIR / "fingerprint.txt"
parts_dir = OUT_DIR / "parts"
if fp_file.exists() and fp_file.read_text() != fingerprint and parts_dir.exists():
    print("Config/selection changed -> clearing old checkpoints")
    shutil.rmtree(parts_dir)
parts_dir.mkdir(exist_ok=True)
fp_file.write_text(fingerprint)

Subset size: 998
category  FVFA  FVRA  RVFA  RVRA
split                           
test        30    26    23    75
train      108   112   123   350
val         28    28    20    75

REAL / FAKE per split:
label  REAL  FAKE
split            
test     75    79
train   350   343
val      75    76


32

## 3 · Load the frozen encoders

In [5]:
import torch, cv2
from transformers import CLIPModel, CLIPProcessor, Wav2Vec2Model, Wav2Vec2FeatureExtractor, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

clip = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_ID)
w2v = Wav2Vec2Model.from_pretrained(W2V_ID).to(device).eval()
w2v_fe = Wav2Vec2FeatureExtractor.from_pretrained(W2V_ID)

_dtype = torch.float16 if device == "cuda" else torch.float32
try:
    asr = pipeline("automatic-speech-recognition", model=WHISPER_ID, dtype=_dtype, device=0 if device == "cuda" else -1)
except TypeError:
    asr = pipeline("automatic-speech-recognition", model=WHISPER_ID, torch_dtype=_dtype, device=0 if device == "cuda" else -1)

face_det = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def _as_tensor(x):
    """transformers versions differ: some return a tensor, some a ModelOutput."""
    if torch.is_tensor(x):
        return x
    for attr in ("pooler_output", "image_embeds", "text_embeds"):
        if hasattr(x, attr) and getattr(x, attr) is not None:
            return getattr(x, attr)
    return x[0]

device: cuda


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (24 outputs left out)

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

## 4 · Per-modality extractors

In [6]:
def read_frames(path, n=N_FRAMES):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(total * 0.1, total * 0.9, n).astype(int) if total >= n else [0]
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, f = cap.read()
        if ok:
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def crop_face(rgb, margin=0.25):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = face_det.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
    h, w = rgb.shape[:2]
    if len(faces) == 0:                                   # fallback: centre square
        s = min(h, w); y0 = (h - s) // 2; x0 = (w - s) // 2
        return rgb[y0:y0 + s, x0:x0 + s], False
    x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
    m = int(margin * max(fw, fh))
    return rgb[max(0, y - m):min(h, y + fh + m), max(0, x - m):min(w, x + fw + m)], True

@torch.no_grad()
def embed_images(imgs):
    inp = clip_proc(images=imgs, return_tensors="pt").to(device)
    e = _as_tensor(clip.get_image_features(pixel_values=inp["pixel_values"]))
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

@torch.no_grad()
def embed_text(texts):
    inp = clip_proc.tokenizer(list(texts), padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
    e = _as_tensor(clip.get_text_features(**inp))
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

def load_audio(path, sr=16000, max_sec=MAX_AUDIO_SEC):
    cmd = ["ffmpeg", "-v", "error", "-i", path, "-t", str(max_sec), "-ac", "1", "-ar", str(sr), "-f", "f32le", "-"]
    out = subprocess.run(cmd, capture_output=True)
    if out.returncode != 0 or len(out.stdout) < 3200:     # < 0.1 s -> treat as missing
        return None
    return np.frombuffer(out.stdout, dtype=np.float32).copy()

@torch.no_grad()
def embed_audio(wav):
    x = w2v_fe(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    hs = w2v(x, output_hidden_states=True).hidden_states            # 13 x (1, T, 768)
    return torch.stack([h.mean(dim=1).squeeze(0) for h in hs]).cpu().numpy()   # (13, 768)

def transcribe(wavs):
    kw = {"task": "transcribe"}
    if WHISPER_LANG:
        kw["language"] = WHISPER_LANG
    outs = asr([{"raw": w, "sampling_rate": 16000} for w in wavs], batch_size=BATCH, generate_kwargs=kw)
    return [o["text"].strip() for o in outs]

## 5 · Main loop (checkpointed)

In [7]:
for start in tqdm(range(0, len(sub), BATCH), desc="batches"):
    pf = parts_dir / f"part_{start:06d}.npz"
    if pf.exists():
        continue
    chunk = sub.iloc[start:start + BATCH]
    img_e, aud_e, wavs, face_rate, ok_img, ok_aud = [], [], [], [], [], []
    for path in chunk.path:
        frames = read_frames(path)
        if frames:
            crops, found = zip(*[crop_face(f) for f in frames])
            e = embed_images(list(crops)).mean(0)
            e = e / (np.linalg.norm(e) + 1e-9)
            face_rate.append(float(np.mean(found))); ok_img.append(1)
        else:
            e = np.zeros(512, np.float32); face_rate.append(0.0); ok_img.append(0)
        img_e.append(e)

        wav = load_audio(path)
        if wav is None:
            aud_e.append(np.zeros((13, 768), np.float32)); wavs.append(np.zeros(16000, np.float32)); ok_aud.append(0)
        else:
            aud_e.append(embed_audio(wav)); wavs.append(wav); ok_aud.append(1)

    texts = transcribe(wavs)
    texts = [t if ok else "" for t, ok in zip(texts, ok_aud)]
    txt_e = embed_text([t if t else " " for t in texts])

    np.savez_compressed(
        pf,
        clip_id=chunk.clip_id.values,
        image=np.stack(img_e).astype(np.float32),
        audio_layers=np.stack(aud_e).astype(np.float32),
        text=txt_e.astype(np.float32),
        transcript=np.array(texts, dtype=str),
        face_rate=np.array(face_rate, np.float32),
        ok_img=np.array(ok_img, np.int8),
        ok_aud=np.array(ok_aud, np.int8),
    )

batches:   0%|          | 0/63 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


## 6 · Merge checkpoints → `features.npz` + `meta.csv`

In [8]:
parts = sorted(parts_dir.glob("part_*.npz"))
loaded = [np.load(p) for p in parts]
cat = lambda k: np.concatenate([z[k] for z in loaded])
order = np.argsort(cat("clip_id"))
get = lambda k: cat(k)[order]

clip_ids = get("clip_id")
assert len(clip_ids) == len(sub) and (clip_ids == sub.clip_id.values).all(), "missing batches — re-run the main loop"

np.savez_compressed(OUT_DIR / "features.npz",
                    clip_id=clip_ids, image=get("image"), audio_layers=get("audio_layers"), text=get("text"))

out_meta = sub.copy()
out_meta["transcript"] = get("transcript")
out_meta["face_rate"] = get("face_rate")
out_meta["ok_img"] = get("ok_img")
out_meta["ok_aud"] = get("ok_aud")
out_meta.to_csv(OUT_DIR / "meta.csv", index=False)

print("saved:", [f.name for f in OUT_DIR.iterdir() if f.is_file()])
print("clips with failed image decode:", int((out_meta.ok_img == 0).sum()), "| failed audio:", int((out_meta.ok_aud == 0).sum()))
print("mean face-detection rate:", round(float(out_meta.face_rate.mean()), 3))

saved: ['meta.csv', 'fingerprint.txt', 'features.npz']
clips with failed image decode: 0 | failed audio: 0
mean face-detection rate: 0.861


## 7 · Quick sanity checks (also useful material for the paper's dataset section)

In [9]:
Z = np.load(OUT_DIR / "features.npz")
sim = (Z["image"] * Z["text"]).sum(1)          # both L2-normalised -> cosine similarity transcript <-> face image
out_meta["clip_txt_img_sim"] = sim
print("Mean CLIP transcript-image cosine similarity per category (a first look at 'semantic mismatch'):")
print(out_meta.groupby("category").clip_txt_img_sim.agg(["mean", "std", "count"]).round(4).to_string())
print("\nSample transcripts:")
for c in ["RVRA", "RVFA", "FVRA", "FVFA"]:
    rows = out_meta[out_meta.category == c].head(2)
    for t in rows.transcript:
        print(f"  [{c}] {t[:110]}")
print("\nEmpty transcripts per category:")
print((out_meta.transcript.fillna("").str.len() == 0).groupby(out_meta.category).sum().to_string())

Mean CLIP transcript-image cosine similarity per category (a first look at 'semantic mismatch'):
            mean     std  count
category                       
FVFA      0.2071  0.0214    166
FVRA      0.2136  0.0271    166
RVFA      0.2096  0.0240    166
RVRA      0.2143  0.0255    500

Sample transcripts:
  [RVRA] us go up and you don't get to see each other. It's like we're downstairs and one guy does his thing and all yo
  [RVRA] all the characters and met everyone. So whatever he told me is what I did.
  [RVFA] No dictating the pace to you know because like I said what I was in there waiting in eight and it just time it
  [RVFA] Well, I mean, inside how what you was kind of relentless and you're shooting crystals at once, this is quite c
  [FVRA] watch it anyway but not being in is quite an ask because you don't know what's going on yeah so actually it wa
  [FVRA] than what they had in Czech world before and better than all the other clubs so in terms
  [FVFA] And he was reasonab

In [10]:
# Optional: bundle everything so it is easy to download
shutil.make_archive("/kaggle/working/features_bundle" if str(OUT_DIR).startswith("/kaggle") else str(OUT_DIR) + "_bundle",
                    "zip", OUT_DIR, ".")  # parts/ included; delete parts/ first if you want it smaller

'/kaggle/working/features_bundle.zip'